# colab_10b — cross-FM zero-shot comparison (Geneformer vs scGPT)

Regime #1 (zero-shot) side-by-side read of the two frozen baselines saved by colab_09 (Geneformer, 768-dim) and colab_10 (scGPT, 512-dim), both embedded from the **same 142,588-cell glia substrate** (54,805 microglia + 87,783 astrocytes), cell-for-cell aligned by `cell_index`.

This notebook adds no new embedding. It (1) tabulates panel to vocabulary coverage for both models, (2) recomputes the silhouette battery in each raw FM space and adds the per-lineage **study** silhouette that neither single-FM notebook computed, and (3) retro-runs the SEA-AD UMAP-Y gradient statistic for **Geneformer** — the one open item, since colab_09 only had a pixel-colour read of its UMAP, not a quantitative measure.

No GPU and no model download: it reads two `.h5ad` baselines plus `outputs/audit_report.json` from Drive.

## 1 — Setup

### 1a — Mount Drive, pull the repo, minimal imports

Comparison-only: no foundation-model install and no checkpoint download. Only scanpy/anndata are pinned on top of Colab's base (matplotlib, scikit-learn and scipy already ship with it).

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print("Python:", sys.version.split()[0])

# Comparison-only notebook: it reads the two saved baselines + the audit trail, so NO FM
# install (no Geneformer, no scGPT, no model download). Only scanpy/anndata are pinned on
# top of Colab's base to match colab_09/colab_10; sklearn/scipy/matplotlib are already present.
!pip install -q "scanpy==1.12.2" "anndata==0.12.19"

## 2 — Environment capture

### 2a — pip freeze + environment JSON

Snapshot the exact resolved stack for the Methods record — the same discipline every notebook in this project follows. Written under `outputs/software_versions/`.

In [ ]:
import json, platform
from datetime import date

TODAY = date.today().isoformat()
SW_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(SW_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(SW_DIR, f"colab_10b_{TODAY}_pip_freeze.txt")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
with open(FREEZE_PATH, "w") as f:
    f.write(freeze)

def _gpu():
    try:
        import torch
        if torch.cuda.is_available():
            return {"cuda": torch.version.cuda, "device": torch.cuda.get_device_name(0)}
    except Exception:
        pass
    return {"cuda": None, "device": "cpu-only (comparison notebook needs no GPU)"}

env = {
    "notebook": "colab_10b_crossfm_zeroshot_comparison",
    "date": TODAY,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": _gpu(),
}
for pkg in ["numpy", "pandas", "scipy", "scanpy", "anndata", "sklearn", "matplotlib"]:
    try:
        mod = __import__(pkg)
        env[pkg] = getattr(mod, "__version__", "unknown")
    except Exception as e:
        env[pkg] = f"IMPORT FAILED: {e}"

ENV_JSON_PATH = os.path.join(SW_DIR, f"colab_10b_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))
print("\nwrote:", FREEZE_PATH, "\n      ", ENV_JSON_PATH)

## 3 — Load both zero-shot baselines

### 3a — Load the Geneformer and scGPT baselines, assert cell-for-cell alignment

Each baseline stores its per-cell embedding in `.X`, its own UMAP in `obsm["X_umap"]`, and the shared label columns in `obs`. Both descend from one substrate, so the cells and their labels must match exactly; the notebook sorts both by `cell_index` and fails loud on any mismatch before comparing row-for-row.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
FIG_DIR = os.path.join(REPO_PATH, "figures", "colab_10b")
os.makedirs(FIG_DIR, exist_ok=True)

GF_PATH    = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
SCGPT_PATH = os.path.join(DRIVE_ROOT, "scgpt", "glia_scgpt_zeroshot.h5ad")
for p in (GF_PATH, SCGPT_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing zero-shot baseline {p} (colab_09 / colab_10 output)")

gf = sc.read_h5ad(GF_PATH)
sg = sc.read_h5ad(SCGPT_PATH)
print("Geneformer baseline:", gf.shape, "| emb dim:", gf.X.shape[1])
print("scGPT      baseline:", sg.shape, "| emb dim:", sg.X.shape[1])

# Both baselines were embedded from the SAME 142,588-cell substrate, cell-for-cell, with a
# stable cell_index. Sort both to cell_index order and fail loud on any mismatch — every
# downstream side-by-side comparison assumes row i is the same cell in both files.
gf = gf[np.argsort(gf.obs["cell_index"].values)].copy()
sg = sg[np.argsort(sg.obs["cell_index"].values)].copy()
assert gf.n_obs == sg.n_obs, f"cell count differs: GF {gf.n_obs} vs scGPT {sg.n_obs}"
assert np.array_equal(gf.obs["cell_index"].values, sg.obs["cell_index"].values), \
    "cell_index order differs between baselines after sort — substrates are not identical"
# The label columns the evals slice on must be identical between the two files.
for col in ["lineage", "study_id", "apoe_carrier", "substate", "donor_id"]:
    assert (gf.obs[col].astype(str).values == sg.obs[col].astype(str).values).all(), \
        f"obs['{col}'] differs between baselines — same substrate should carry identical labels"
print("\nalignment OK — identical", gf.n_obs, "cells + identical labels in both baselines")
print("lineage :", gf.obs["lineage"].value_counts().to_dict())
print("study_id:", gf.obs["study_id"].value_counts().to_dict())

# Shared label frame (identical in both) + a registry of the two embeddings to loop over.
obs = gf.obs
EMB = {
    "Geneformer": {"X": np.asarray(gf.X), "umap": gf.obsm["X_umap"], "dim": gf.X.shape[1]},
    "scGPT":      {"X": np.asarray(sg.X), "umap": sg.obsm["X_umap"], "dim": sg.X.shape[1]},
}

## 4 — Vocabulary coverage: Geneformer vs scGPT

### 4a — Panel to vocabulary coverage and niche-gene survival for both models

Before a model can embed a gene it must be able to tokenize it. Each FM audited its own panel to vocabulary coverage (and hard-gated on APOE) in its own notebook and wrote the result to the audit trail; recomputing here would mean re-downloading both vocabularies, so this reads the two audited entries and lays them side by side. The load-bearing column is niche-gene `in_vocab`: the eight genes the whole astro–microglia APOE niche rests on must survive tokenization in both models.

In [ ]:
AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

NICHE_CRITICAL_GENES = ["APOE", "TREM2", "MS4A6A", "CLU", "GFAP", "AQP4", "AIF1", "CSF1R"]
FM_KEYS = {"Geneformer": "geneformer_zeroshot", "scGPT": "scgpt_zeroshot"}

# Vocab coverage was audited (and gated on APOE) in each FM's own notebook and written to the
# audit trail. Recomputing it here would mean re-downloading both FM vocabularies; the
# audit_report entries are the pre-registered artifacts, so read + tabulate them side by side.
va = {}
for fm, key in FM_KEYS.items():
    entry = report.get(key)
    if entry is None or "vocab_audit" not in entry:
        raise KeyError(f"audit_report missing vocab_audit for {key} — run colab_09/colab_10 first")
    va[fm] = entry["vocab_audit"]

n_panel = va["Geneformer"]["n_genes_panel"]
assert va["scGPT"]["n_genes_panel"] == n_panel, "panel size differs between FM audits — not the same panel"

cov = pd.DataFrame({
    fm: {
        "vocab_size":     va[fm]["vocab_size"],
        "genes_in_vocab": va[fm]["n_in_vocab"],
        "panel_size":     va[fm]["n_genes_panel"],
        "frac_in_vocab":  va[fm]["frac_in_vocab"],
        "apoe_gate":      va[fm]["apoe_hard_fail_gate"],
    } for fm in FM_KEYS
}).T
print(f"Panel -> FM-vocabulary coverage (shared panel = {n_panel} genes):")
print(cov.to_string())

# Niche-critical gene survival, both FMs side by side (in_vocab is the load-bearing column).
rows = []
for g in NICHE_CRITICAL_GENES:
    row = {"gene": g}
    for fm in FM_KEYS:
        st = va[fm]["niche_status"].get(g, {})
        row[f"{fm}_in_vocab"] = st.get("in_vocab")
    rows.append(row)
niche_tbl = pd.DataFrame(rows).set_index("gene")
print("\nNiche-critical gene survival (in_vocab):")
print(niche_tbl.to_string())

## 5 — Silhouette battery: what each raw FM space encodes

### 5a — Reproduce each FM's silhouettes and add the per-lineage study silhouette

A silhouette near the ca. 0.02 floor means a label has no clean geometry in that space; a clearly higher value means the space separates that label. Both FMs' six original metrics are recomputed here (and checked against the stored values as a reproducibility guard), plus two new ones — `study_micro` and `study_astro`. The pooled `study_all` silhouette can read near-zero while real study structure still sits **within** each lineage; the per-lineage study silhouette is the orientation-free measure of that residual batch structure, directly comparable across the two models.

In [ ]:
from sklearn.metrics import silhouette_score

APOE_EXCLUDE_VALS = {"e2"}   # E2-without-E4 is off-contract (EVALUATION_CONTRACT); not scored as a class

def _sil(X, mask, label, exclude_vals=None):
    keep = mask & obs[label].notna().values
    if exclude_vals:
        keep = keep & ~obs[label].astype(str).isin(exclude_vals).values
    y  = obs[label].astype(str).values[keep]
    Xk = X[keep]
    if len(set(y)) < 2 or Xk.shape[0] < 50:
        return None
    n = min(10000, Xk.shape[0])
    idx = np.random.default_rng(0).choice(Xk.shape[0], n, replace=False)
    return round(float(silhouette_score(Xk[idx], y[idx])), 4)

all_mask   = np.ones(obs.shape[0], bool)
micro_mask = (obs["lineage"] == "microglia").values
astro_mask = (obs["lineage"] == "astrocyte").values

BATTERY = {
    "lineage_all":    (all_mask,   "lineage",      None),   # expect HIGH — identity saturates
    "study_all":      (all_mask,   "study_id",     None),   # pooled: can hide within-lineage structure
    "study_micro":    (micro_mask, "study_id",     None),   # NEW — per-lineage study residual (contract-flagged gap)
    "study_astro":    (astro_mask, "study_id",     None),   # NEW
    "apoe_micro":     (micro_mask, "apoe_carrier", APOE_EXCLUDE_VALS),
    "apoe_astro":     (astro_mask, "apoe_carrier", APOE_EXCLUDE_VALS),
    "substate_micro": (micro_mask, "substate",     None),
    "substate_astro": (astro_mask, "substate",     None),
}

sil = {fm: {} for fm in EMB}
for fm, e in EMB.items():
    for name, (m, lab, ex) in BATTERY.items():
        sil[fm][name] = _sil(e["X"], m, lab, ex)

sil_tbl = pd.DataFrame(sil)
print("Silhouettes (zero-shot embedding space) — Euclidean, 10k sample, seed 0:\n")
print(sil_tbl.to_string())

# Reproducibility guard: the six metrics each notebook already stored must match here (same
# helper, same seed, same data). Fail loud if a loaded baseline no longer reproduces its own
# recorded silhouettes — that would mean the file or the metric drifted.
STORED = {fm: report[FM_KEYS[fm]]["zeroshot_silhouettes"] for fm in EMB}
mism = []
for fm in EMB:
    for name, v in STORED[fm].items():
        got = sil[fm].get(name)
        if got is None or abs(got - v) > 5e-3:
            mism.append(f"{fm}/{name}: recomputed {got} vs stored {v}")
if mism:
    raise AssertionError("silhouette reproduction drift:\n  " + "\n  ".join(mism))
print("\nreproduction OK — all stored silhouettes reproduced within 5e-3")

# grouped comparison figure
order = list(BATTERY)
xg = np.arange(len(order)); w = 0.38
fig, axf = plt.subplots(figsize=(11, 5))
for i, fm in enumerate(EMB):
    vals = [sil[fm][k] if sil[fm][k] is not None else 0.0 for k in order]
    axf.bar(xg + (i - 0.5) * w, vals, w, label=fm)
axf.axhline(0.02, ls="--", lw=1, color="grey", label="0.02 floor (no clean geometry)")
axf.axhline(0.0, lw=0.8, color="k")
axf.set_xticks(xg); axf.set_xticklabels(order, rotation=45, ha="right")
axf.set_ylabel("silhouette"); axf.set_title("Zero-shot silhouette battery — Geneformer vs scGPT")
axf.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "5a_silhouette_battery_crossfm.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6 — SEA-AD spatial gradient: the Geneformer retro-run

### 6a — Per-lineage SEA-AD fraction across UMAP-Y quintiles, both FMs

Within each lineage, bin cells into fifths along UMAP-Y and measure the SEA-AD fraction per bin: a monotone staircase means SEA-AD occupies a distinct region of the map rather than mixing in. colab_10 computed this for scGPT; here it is computed for **Geneformer** for the first time (colab_09 relied on classifying UMAP pixel colours). The UMAP orientation is arbitrary and independent per model, so the **sign** of Spearman rho is not comparable across FMs — only `|rho|`, the quintile spread, and the monotonicity are. The figure orients every staircase low to high SEA-AD so the shapes line up.

In [ ]:
from scipy.stats import spearmanr

def _study_gradient(umap, mask, n_bins=5):
    y = umap[mask, 1]
    is_sea = (obs.loc[mask, "study_id"].astype(str) == "SEA-AD").values.astype(int)
    bins = pd.qcut(y, n_bins, labels=False, duplicates="drop")
    frac = pd.Series(is_sea).groupby(bins).mean()
    counts = pd.Series(is_sea).groupby(bins).size()
    rho, p = spearmanr(y, is_sea)
    return frac, counts, rho, p

# UMAP-Y sign/orientation is arbitrary and independent per FM (each notebook ran its own UMAP),
# so the SIGN of rho is NOT comparable across FMs — only |rho|, the quintile spread, and the
# monotonicity are. This retro-runs the scGPT check from colab_10 and, for the first time,
# computes the same statistic for Geneformer (colab_09 only had a PNG pixel read).
print("SEA-AD spatial gradient — UMAP-Y quintiles (bin 0 = bottom ... bin 4 = top of each FM's own UMAP)\n")
grad = {}
for fm, e in EMB.items():
    print(f"=== {fm} (dim {e['dim']}) ===")
    for lin in ["microglia", "astrocyte"]:
        m = (obs["lineage"] == lin).values
        frac, counts, rho, p = _study_gradient(e["umap"], m)
        overall = float((obs.loc[m, "study_id"].astype(str) == "SEA-AD").mean())
        spread  = float(frac.max() - frac.min())
        grad[(fm, lin)] = {"overall": overall, "spread": spread,
                           "rho": float(rho), "abs_rho": abs(float(rho)), "p": float(p)}
        print(f"  {lin:10s} overall SEA-AD={overall:.3f}  spread={spread:.3f}  "
              f"rho={rho:+.3f} |rho|={abs(rho):.3f}  p={p:.2e}")
        print("     quintile fracs:", "  ".join(f"{frac[b]:.3f}" for b in frac.index))
    print()

grad_tbl = pd.DataFrame(grad).T
grad_tbl.index.names = ["FM", "lineage"]
print(grad_tbl[["overall", "spread", "abs_rho", "rho", "p"]].to_string())

# side-by-side quintile-staircase figure (per lineage), oriented low->high SEA-AD so the
# shapes are comparable across FMs despite the arbitrary per-FM UMAP orientation.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, lin in zip(axes, ["microglia", "astrocyte"]):
    m = (obs["lineage"] == lin).values
    for fm, e in EMB.items():
        frac, counts, rho, p = _study_gradient(e["umap"], m)
        fv = frac.values
        if fv[0] > fv[-1]:                 # orient low->high SEA-AD for a fair shape comparison
            fv = fv[::-1]
        ax.plot(range(len(fv)), fv, marker="o",
                label=f"{fm} (|rho|={abs(rho):.2f}, spread={frac.max()-frac.min():.2f})")
    ax.set_title(lin); ax.set_xlabel("UMAP-Y quintile (oriented low->high SEA-AD)")
    ax.set_ylim(-0.02, 1.02)
axes[0].set_ylabel("SEA-AD fraction"); axes[0].legend()
fig.suptitle("SEA-AD fraction across UMAP-Y quintiles — Geneformer vs scGPT")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "6a_seaad_gradient_crossfm.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7 — Save + handoff

### 7a — Append the comparison to the audit trail and print the commit commands

Writes a `crossfm_zeroshot_comparison` block (coverage, silhouettes, gradient stats) into `outputs/audit_report.json` and prints the WSL git commands. No new `.h5ad` — the two baselines are unchanged; this notebook only reports over them.

In [ ]:
import shlex

report["crossfm_zeroshot_comparison"] = {
    "status": "computed",
    "date": TODAY,
    "notebook": "colab_10b_crossfm_zeroshot_comparison",
    "n_cells": int(obs.shape[0]),
    "compared": {
        "Geneformer": {"emb_dim": int(EMB["Geneformer"]["dim"]), "frac_in_vocab": va["Geneformer"]["frac_in_vocab"]},
        "scGPT":      {"emb_dim": int(EMB["scGPT"]["dim"]),      "frac_in_vocab": va["scGPT"]["frac_in_vocab"]},
    },
    "silhouettes": {fm: sil[fm] for fm in EMB},
    "seaad_gradient": {f"{fm}|{lin}": grad[(fm, lin)] for (fm, lin) in grad},
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL — Colab has no git creds) ===")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add "
      + " ".join(shlex.quote(r) for r in rel) + " figures/colab_10b")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
      "'colab_10b: cross-FM zero-shot comparison (Geneformer vs scGPT)'")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")

### Carried forward

This closes the regime #1 zero-shot comparison (the last open item from colab_10). Next in the plan: the continued-pretraining (CPT) notebooks (`colab_11+`), where these two frozen baselines become the reference each post-CPT embedding is scored against (detector #1 drift + evals #1/#2), aligned by `cell_index`.